# XFAL ETL — Remote PostgreSQL

Same normalization as the local notebook, targeting a remote Postgres host
(for example AWS RDS). Configure credentials via environment variables — never hardcode secrets.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
df = pd.read_csv(ROOT / "Data" / "Data.csv")

df["company_id"] = (
    pd.factorize(
        df["company_name"].astype(str)
        + df["address"].astype(str)
        + df["city"].astype(str)
        + df["state"].astype(str)
        + df["zip"].astype(str)
    )[0]
    + 1
)
df["department_id"] = pd.factorize(df["department"].astype(str))[0] + 1

df_company = df[["company_id", "company_name", "address", "city", "state", "zip"]].drop_duplicates()
df_department = (
    df[["department_id", "department"]]
    .drop_duplicates()
    .rename(columns={"department": "department_name"})
)
df_employee = df[["first_name", "last_name", "phone1", "phone2", "email", "company_id", "department_id"]]
df.head()

In [ ]:
import os
import psycopg2
from psycopg2 import extras

host = os.getenv("REMOTE_POSTGRES_HOST")
if not host:
    raise ValueError("Set REMOTE_POSTGRES_HOST (and related REMOTE_POSTGRES_* vars) before running.")

connection = psycopg2.connect(
    dbname=os.getenv("REMOTE_POSTGRES_DB", "xfal"),
    user=os.getenv("REMOTE_POSTGRES_USER", "postgres"),
    password=os.getenv("REMOTE_POSTGRES_PASSWORD", ""),
    host=host,
    port=os.getenv("REMOTE_POSTGRES_PORT", "5432"),
)
print(f"Connected to remote host {host}")

In [ ]:
def insert_into_table(connection, df, table_name):
    tuples = [tuple(x) for x in df.to_numpy()]
    cols = ",".join(list(df.columns))
    query = f"INSERT INTO {table_name} ({cols}) VALUES %s ON CONFLICT DO NOTHING"
    cursor = connection.cursor()
    try:
        extras.execute_values(cursor, query, tuples, template=None)
        connection.commit()
        print(f"Inserted into {table_name}: {len(tuples)} rows")
    except psycopg2.Error as e:
        connection.rollback()
        print(f"Error inserting into {table_name}: {e}")
    finally:
        cursor.close()

insert_into_table(connection, df_company, "company")
insert_into_table(connection, df_department, "department")
insert_into_table(connection, df_employee, "employee")
connection.close()